# Experiment 5: CGM + time_since_meal + image (clip, gated)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+gated_clip (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'gated_clip'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_clip'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_clip_stability_summary.json'

In [4]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [7]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=768, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_clip' /f'gated_cgm_time_clip_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_clip_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — CGM+time+clip (gated) | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.40 | MAE 9.10 | gRMSE 15.58 | Zone A 95.24% | A+B 99.57%
 30 min | RMSE 19.95 | MAE 13.20 | gRMSE 23.95 | Zone A 89.56% | A+B 99.36%
 60 min | RMSE 28.49 | MAE 18.57 | gRMSE 35.42 | Zone A 80.73% | A+B 98.69%
 90 min | RMSE 34.23 | MAE 22.21 | gRMSE 43.38 | Zone A 74.65% | A+B 97.91%
120 min | RMSE 37.45 | MAE 24.55 | gRMSE 47.93 | Zone A 70.83% | A+B 97.18%

--------------------------------------------------------------------------------
Test results — CGM+time+clip (gated) | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.68 | MAE 7.84 | gRMSE 13.36 | Zone A 97.09% | A+B 99.78%
 30 min | RMSE 19.52 | MAE 12.83 | gRMSE 23.35 | Zone A 89.67% | A+B 99.54%
 60 min | RMSE 29.96 | MAE 19.37 | gRMSE 37.39 | Zone A 79.11% | A+B 98.62%
 90 min | RMSE 35.83 | MAE 23.47 | gRMSE 45.33 | Zone A 72.62% | A+B 97.76%
120 min | RMSE 38.83 | MAE 25.66 | gRMSE 49.49 | Zone A 69.27% | A+B 97.35%

--------------------------------------------------------------------------------
Test results — CGM+time+clip (gated) | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


...

 15 min | RMSE 11.72 | MAE 7.93 | gRMSE 13.12 | Zone A 96.42% | A+B 99.74%
 30 min | RMSE 18.40 | MAE 12.19 | gRMSE 21.50 | Zone A 90.55% | A+B 99.52%
 60 min | RMSE 27.49 | MAE 17.93 | gRMSE 33.88 | Zone A 81.45% | A+B 98.91%
 90 min | RMSE 33.54 | MAE 21.79 | gRMSE 42.36 | Zone A 75.14% | A+B 98.14%
120 min | RMSE 37.02 | MAE 24.08 | gRMSE 47.27 | Zone A 71.88% | A+B 97.47%

--------------------------------------------------------------------------------
Test results — CGM+time+clip (gated) | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.99 | MAE 8.64 | gRMSE 14.44 | Zone A 95.20% | A+B 99.61%
 30 min | RMSE 19.44 | MAE 12.74 | gRMSE 22.67 | Zone A 89.71% | A+B 99.44%
 60 min | RMSE 28.23 | MAE 18.35 | gRMSE 34.76 | Zone A 81.05% | A+B 98.75%
 90 min | RMSE 34.15 | MAE 22.11 | gRMSE 43.20 | Zone A 74.84% | A+B 97.94%
120 min | RMSE 37.47 | MAE 24.52 | gRMSE 47.93 | Zone A 71.16% | A+B 97.13%

--------------------------------------------------------------------------------
Test results — CGM+time+clip (gated) | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.83 | MAE 8.51 | gRMSE 14.46 | Zone A 95.52% | A+B 99.63%
 30 min | RMSE 19.42 | MAE 12.70 | gRMSE 22.86 | Zone A 90.07% | A+B 99.46%
 60 min | RMSE 28.19 | MAE 18.29 | gRMSE 34.74 | Zone A 81.35% | A+B 98.74%
 90 min | RMSE 33.85 | MAE 21.89 | gRMSE 42.66 | Zone A 75.56% | A+B 98.04%
120 min | RMSE 37.14 | MAE 24.30 | gRMSE 47.31 | Zone A 71.70% | A+B 97.45%

--------------------------------------------------------------------------------


In [8]:
# Test Results:
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [r['test_results'][h]['ceg']['A'] for r in all_results]
        elif m == 'zone_ab':
            seed_values = [r['test_results'][h]['ceg']['A'] + r['test_results'][h]['ceg']['B']
                            for r in all_results]
        else:
            seed_values = [r['test_results'][h][m] for r in all_results]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [_extract(r['test_results_per_participant'][pid][h], m)
                                    for r in all_results]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}

print(f'=== {MODEL_NAME} test set: overall mean +/- SD_seed (SD_participant) ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME, 'seeds': SEEDS, 'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS, 'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')

=== CGM+time+clip (gated) test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 12.67 +/- 0.58 (2.21) mg/dL   MAE = 8.48 +/- 0.40 (1.69)   gRMSE = 14.34 +/- 0.69 (3.74)   Zone A = 95.76 +/- 0.76 (2.98)%   Zone A+B = 99.65 +/- 0.08 (0.62)%
 30-min:  RMSE = 19.46 +/- 0.42 (4.42) mg/dL   MAE = 12.80 +/- 0.26 (3.03)   gRMSE = 22.97 +/- 0.72 (7.55)   Zone A = 89.75 +/- 0.34 (4.09)%   Zone A+B = 99.46 +/- 0.05 (0.76)%
 60-min:  RMSE = 28.56 +/- 0.79 (9.87) mg/dL   MAE = 18.55 +/- 0.45 (6.08)   gRMSE = 35.31 +/- 1.15 (15.83)   Zone A = 80.66 +/- 0.86 (5.05)%   Zone A+B = 98.74 +/- 0.08 (1.30)%
 90-min:  RMSE = 34.33 +/- 0.79 (13.88) mg/dL   MAE = 22.32 +/- 0.60 (8.27)   gRMSE = 43.35 +/- 1.02 (21.89)   Zone A = 74.50 +/- 1.01 (6.77)%   Zone A+B = 97.99 +/- 0.12 (2.47)%
120-min:  RMSE = 37.53 +/- 0.65 (16.40) mg/dL   MAE = 24.64 +/- 0.57 (9.79)   gRMSE = 47.85 +/- 0.82 (25.68)   Zone A = 70.90 +/- 1.05 (8.06)%   Zone A+B = 97.39 +/- 0.16 (3.75)%

saved per-seed results to /c